In [ ]:
# ルートに移動

%cd ..

In [ ]:
# ライブラリのインポート

import glob
import json
import os
import random
import shutil

import numpy as np
import yaml
from PIL import Image
from ultralytics.data.converter import convert_coco
from ultralytics.utils.ops import ltwh2xyxy, xyxy2xywhn

In [ ]:
# ディレクトリの定義

generated_dir = os.path.join("data", "generated")
prepared_dir = os.path.join("data", "prepared")
raw_dir = os.path.join("data", "raw")
segmented_dir = os.path.join("data", "segmented")

In [ ]:
# データセットの準備

convert_coco(
    labels_dir=os.path.join(raw_dir, "annotations"),
    save_dir=prepared_dir,
    cls91to80=False,
)

os.makedirs(os.path.join(prepared_dir, "images", "train"), exist_ok=True)

annotation_path = os.path.join(raw_dir, "annotations", "train.json")
with open(annotation_path, "r") as f:
    data = json.load(f)

for image in data["images"]:
    file_name = image["file_name"]

    src = os.path.join(raw_dir, "images", file_name)
    dst = os.path.join(prepared_dir, "images", "train", file_name)

    shutil.copy2(src, dst)

names = {cat["id"] - 1: cat["name"] for cat in data["categories"]}

save_path = os.path.join(prepared_dir, "data.yaml")
with open(save_path, "w") as f:
    yaml.safe_dump(
        {
            "path": prepared_dir,
            "train": os.path.join("images", "train"),
            "val": os.path.join("images", "train"),
            "names": names,
        },
        f,
        sort_keys=False,
    )

In [ ]:
# データセットの拡張

random.seed(0)

annotations = {ann["id"]: ann for ann in data["annotations"]}

foregrounds = sorted(
    glob.glob(os.path.join(segmented_dir, "*.png")),
    key=lambda p: int(os.path.basename(p)[1:-4]),
)
backgrounds = sorted(
    glob.glob(os.path.join(generated_dir, "*.jpg")),
    key=lambda p: int(os.path.basename(p)[1:-4]),
)

margin = 12


def is_oversize(img1: Image.Image, img2: Image.Image):
    return (
        img2.width + margin * 2 > img1.width or img2.height + margin * 2 > img1.height
    )


def is_overlap(box1: tuple, box2: tuple):
    return not (
        box1[2] < box2[0] or box1[0] > box2[2] or box1[3] < box2[1] or box1[1] > box2[3]
    )


probs = {1: 0.15, 2: 0.15, 3: 0.25, 4: 0.25, 5: 0.20}
population, weights = zip(*sorted(probs.items()))


def sample():
    return random.choices(population, weights, k=1)[0]


count = 1

for background in backgrounds:
    bg = Image.open(background).convert("RGBA")
    boxes = []
    anns = []

    for _ in range(sample()):
        foreground = random.choice(foregrounds)
        fg = Image.open(foreground).convert("RGBA")

        if is_oversize(bg, fg):
            scale = min(
                bg.width / (fg.width + margin * 2), bg.height / (fg.height + margin * 2)
            )
            scale = scale * random.uniform(0.8, 1.0)
            fg = fg.resize(
                (int(fg.width * scale), int(fg.height * scale)), Image.LANCZOS
            )

        for _ in range(100):
            x = random.randint(0, bg.width - fg.width - margin * 2) + margin
            y = random.randint(0, bg.height - fg.height - margin * 2) + margin
            box = (x, y, x + fg.width, y + fg.height)
            if all(not is_overlap(box, b) for b in boxes):
                bg.alpha_composite(fg, (x, y))
                boxes.append(box)

                file_name = os.path.basename(foreground)
                id = int(file_name[1:-4])
                category_id = annotations[id]["category_id"]

                anns.append(
                    {"category_id": category_id, "bbox": [x, y, fg.width, fg.height]}
                )

                break

    assert len(anns) > 0

    image = bg.convert("RGB")
    save_path = os.path.join(prepared_dir, "images", "train", f"A{count}.jpg")
    image.save(save_path)

    save_path = os.path.join(prepared_dir, "labels", "train", f"A{count}.txt")
    with open(save_path, "w") as f:
        for ann in anns:
            category_id = int(ann["category_id"])
            class_id = category_id - 1
            xyxy = ltwh2xyxy(np.array([ann["bbox"]], dtype=np.float32))
            xywh = xyxy2xywhn(xyxy, w=bg.width, h=bg.height, clip=True)
            x, y, w, h = xywh[0]
            f.write(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

    count += 1